# 31 — CIL Training V2 (reemplaza Notebook 30)

Entrenamiento del modelo CIL (Conditional Imitation Learning) para el controlador **V2.0**.  
Dataset externo descargado desde GitHub — dos partes combinadas.  

**Archivo exportado:** `cil_model2.pt` → copiar a `models/` del proyecto Webots.  

### Arquitectura
- Backbone DAVE-2 con BatchNorm y `padding=2`
- `AdaptiveAvgPool2d(4, 8)` → flat 2 048 features
- Capas compartidas FC(2048→512→256)
- 4 ramas MLP independientes (una por nav_command)
- Salida: `Tanh() × 0.7` — rango ±0.70 rad

### Compatibilidad con V2.0
El controlador pasa `nav_command` como `torch.Tensor([cmd], dtype=long)` — compatible con `torch.jit.trace`.

---
## 1. Clonar dataset desde GitHub

In [ ]:
!git clone https://github.com/Joseluiscruzvalencia92/CIL_data.git
import os
!ls CIL_data/
!ls CIL_data/PART2/

---
## 2. Instalar dependencias

In [ ]:
!pip install torch torchvision pandas numpy opencv-python-headless matplotlib scikit-learn -q

---
## 3. Verificar GPU

> **Antes de continuar:** en Colab ve a  
> `Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware → GPU (T4 o A100)`  
> y reinicia el entorno. Esta celda lanza un error si no detecta GPU.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU no detectada. En Colab: Entorno de ejecucion → '
        'Cambiar tipo de entorno de ejecucion → '
        'Acelerador de hardware → GPU (T4 o A100).'
    )

device   = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
mem_gb   = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print(f'GPU      : {gpu_name}')
print(f'Memoria  : {mem_gb:.1f} GB')
print(f'Device   : {device}')

---
## 4. Combinar los dos datasets

Las dos partes del dataset tienen `images/` separadas.  
Se agrega la ruta completa a `image_filename` antes de concatenar para evitar colisiones de nombres.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

CSV_1    = 'CIL_data/measurements.csv'
CSV_2    = 'CIL_data/PART2/measurements.csv'
IMAGES_1 = 'CIL_data/images'
IMAGES_2 = 'CIL_data/PART2/images'

print(f'CSV_1 existe    : {os.path.isfile(CSV_1)}')
print(f'CSV_2 existe    : {os.path.isfile(CSV_2)}')
print(f'IMAGES_1 existe : {os.path.isdir(IMAGES_1)}')
print(f'IMAGES_2 existe : {os.path.isdir(IMAGES_2)}')

df1 = pd.read_csv(CSV_1)
df2 = pd.read_csv(CSV_2)

print(f'\nPART 1: {len(df1)} imagenes')
print(f'PART 2: {len(df2)} imagenes')

df1['image_filename'] = df1['image_filename'].apply(
    lambda f: os.path.join(IMAGES_1, f))
df2['image_filename'] = df2['image_filename'].apply(
    lambda f: os.path.join(IMAGES_2, f))

df = pd.concat([df1, df2]).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Total combinado : {len(df)} imagenes')
print(f'\nColumnas        : {list(df.columns)}')

---
## 5. Distribución de comandos y steering

Verificar que los 4 comandos están presentes y que la distribución de steering angle es razonable.

In [ ]:
CMD_NAMES = {0: 'LANE_FOLLOW', 1: 'STRAIGHT', 2: 'LEFT', 3: 'RIGHT'}

counts = df['nav_command'].value_counts().sort_index()
print('Distribucion de nav_command:')
for cmd, count in counts.items():
    print(f'  {cmd} ({CMD_NAMES.get(cmd, "?")}): {count} imagenes')

print(f'\nSteering angle:')
print(df['steering_angle'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar([CMD_NAMES.get(c, str(c)) for c in counts.index], counts.values,
            color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'])
axes[0].set_title('Distribucion por nav_command')
axes[0].set_ylabel('Cantidad de imagenes')
axes[1].hist(df['steering_angle'], bins=50, color='#2196F3', edgecolor='white')
axes[1].set_title('Distribucion de steering angle')
axes[1].set_xlabel('Angulo (rad)')
plt.tight_layout()
plt.show()

---
## 6. Balancear el dataset

LANE_FOLLOW domina el dataset crudo.  
Se limita a `max(n_intersecciones × 3, 2000)` muestras para que los comandos de intersección
tengan peso razonable en cada época sin perder todo el seguimiento de carril.

In [ ]:
df_intersections = df[df['nav_command'] != 0]
df_lane_follow   = df[df['nav_command'] == 0]

n_intersections = len(df_intersections)
max_lane_follow = max(n_intersections * 3, 2000)

if len(df_lane_follow) > max_lane_follow:
    df_lane_follow = df_lane_follow.sample(max_lane_follow, random_state=42)
    print(f'LANE_FOLLOW reducido a {max_lane_follow}')
else:
    print(f'LANE_FOLLOW mantenido completo: {len(df_lane_follow)}')

df_balanced = pd.concat([df_lane_follow, df_intersections]).reset_index(drop=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nDataset balanceado: {len(df_balanced)} imagenes')
print(df_balanced['nav_command'].value_counts().sort_index())

---
## 7. Dataset PyTorch — `CILDataset`

Pipeline de preprocesamiento (debe coincidir exactamente con el controlador V2.0):
```
BGR  →  RGB  →  Crop [40%h : 95%h]  →  Resize 320×160  →  float32/255  →  ImageNet norm
```

**Augmentaciones** (solo en entrenamiento):
| Tipo | Probabilidad | Nota |
|------|-------------|------|
| Flip horizontal | 50 % | Invierte ángulo y LEFT↔RIGHT |
| Brillo aleatorio | 50 % | Factor ∈ [0.7, 1.3] |
| Ruido gaussiano | 30 % | σ = 8 px |


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2
import random

IMG_H     = 160
IMG_W     = 320
CROP_TOP  = 0.40
CROP_BOT  = 0.95
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]


class CILDataset(Dataset):
    def __init__(self, df, augment=True):
        self.df        = df.reset_index(drop=True)
        self.augment   = augment
        self.normalize = T.Normalize(mean=NORM_MEAN, std=NORM_STD)

    def __len__(self):
        return len(self.df)

    def _load_and_crop(self, filepath):
        img = cv2.imread(filepath)
        if img is None:
            return np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        y1   = int(h * CROP_TOP)
        y2   = int(h * CROP_BOT)
        img  = img[y1:y2, :]
        img  = cv2.resize(img, (IMG_W, IMG_H))
        return img

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        img     = self._load_and_crop(row['image_filename'])
        angle   = float(row['steering_angle'])
        nav_cmd = int(row['nav_command'])

        if self.augment:
            if random.random() < 0.5:
                img     = cv2.flip(img, 1)
                angle   = -angle
                if nav_cmd == 2:
                    nav_cmd = 3
                elif nav_cmd == 3:
                    nav_cmd = 2
            if random.random() < 0.5:
                factor = random.uniform(0.7, 1.3)
                img    = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
            if random.random() < 0.3:
                noise = np.random.normal(0, 8, img.shape).astype(np.int16)
                img   = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

        t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        t = self.normalize(t)
        return t, torch.tensor(nav_cmd, dtype=torch.long), torch.tensor(angle, dtype=torch.float32)


print('CILDataset definido correctamente')

---
## 8. Split train/val y DataLoaders

Split estratificado 80/20 por `nav_command` para que todos los comandos aparezcan en ambos conjuntos.  
`pin_memory=True` activo para transferencias CPU→GPU más rápidas.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df_balanced, test_size=0.2, random_state=42,
    stratify=df_balanced['nav_command'])

train_dataset = CILDataset(df_train, augment=True)
val_dataset   = CILDataset(df_val,   augment=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} muestras  |  {len(train_loader)} batches')
print(f'Val  : {len(val_dataset)} muestras  |  {len(val_loader)} batches')
print(f'Device activo: {device}')

---
## 9. Definir modelo CIL — `CILModel`

```
Imagen (3 × 160 × 320)
   │
   ▼  ── CNN Backbone (DAVE-2 + BatchNorm + padding) ──────────────
   Conv(3→24,  5×5 s=2 p=2) → BN → ELU
   Conv(24→36, 5×5 s=2 p=2) → BN → ELU
   Conv(36→48, 5×5 s=2 p=2) → BN → ELU
   Conv(48→64, 3×3 s=1 p=1) → BN → ELU
   Conv(64→64, 3×3 s=1 p=1) → BN → ELU
   AdaptiveAvgPool2d(4, 8) → Flatten → 2 048 features
   │
   ▼  ── Capas compartidas ─────────────────────────────────────────
   FC(2048→512) → Dropout(0.3) → ELU
   FC(512→256)  → Dropout(0.3) → ELU
   │
   ├── Rama 0  LANE_FOLLOW ──▶ FC(256→128)→ELU → FC(128→64)→ELU → FC(64→1)→Tanh × 0.7
   ├── Rama 1  STRAIGHT    ──▶ FC(256→128)→ELU → FC(128→64)→ELU → FC(64→1)→Tanh × 0.7
   ├── Rama 2  LEFT        ──▶ FC(256→128)→ELU → FC(128→64)→ELU → FC(64→1)→Tanh × 0.7
   └── Rama 3  RIGHT       ──▶ FC(256→128)→ELU → FC(128→64)→ELU → FC(64→1)→Tanh × 0.7
```

La selección de rama se hace con `gather` sobre las 4 salidas apiladas — todos los pesos reciben gradientes.

In [ ]:
class CILModel(nn.Module):
    """
    Conditional Imitation Learning model.
    CNN extrae features; 4 ramas MLP — una por nav_command.
    Exportado con torch.jit.trace — compatible con controlador V2.0.
    """
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(3,  24, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(24), nn.ELU(),
            nn.Conv2d(24, 36, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(36), nn.ELU(),
            nn.Conv2d(36, 48, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(48), nn.ELU(),
            nn.Conv2d(48, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64), nn.ELU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64), nn.ELU(),
            nn.AdaptiveAvgPool2d((4, 8)),
            nn.Flatten(),
        )

        self.shared = nn.Sequential(
            nn.Linear(2048, 512), nn.Dropout(0.3), nn.ELU(),
            nn.Linear(512,  256), nn.Dropout(0.3), nn.ELU(),
        )

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(256, 128), nn.ELU(),
                nn.Linear(128, 64),  nn.ELU(),
                nn.Linear(64, 1),    nn.Tanh(),
            )
            for _ in range(4)
        ])

    def forward(self, image, nav_command):
        features       = self.cnn(image)
        shared         = self.shared(features)
        branch_outputs = torch.stack(
            [branch(shared) for branch in self.branches], dim=1
        )  # (B, 4, 1)
        idx    = nav_command.view(-1, 1, 1).expand(-1, 1, 1)
        output = branch_outputs.gather(1, idx).squeeze(-1).squeeze(-1)  # (B,)
        return output * 0.7


model = CILModel().to(device)

dummy_img = torch.randn(4, 3, IMG_H, IMG_W, device=device)
dummy_cmd = torch.tensor([0, 1, 2, 3], dtype=torch.long, device=device)
dummy_out = model(dummy_img, dummy_cmd)
print(f'Device   : {next(model.parameters()).device}')
print(f'Output shape : {dummy_out.shape}  (expected [4])')
print(f'Parametros   : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

---
## 10. Entrenamiento

- **Loss:** MSE sobre ángulo de steering (radianes)
- **Optimizador:** Adam, `lr=1e-3`, `weight_decay=1e-4`
- **Scheduler:** `ReduceLROnPlateau` — divide LR a la mitad tras 4 épocas sin mejora
- **Gradient clipping:** `max_norm=1.0`
- **Early stopping:** paciencia de 8 épocas
- El mejor modelo se guarda en `cil_model2_best.pth`

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

EPOCHS   = 50
LR       = 1e-3
PATIENCE = 8

optimizer        = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler        = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
criterion        = nn.MSELoss()
best_val_loss    = float('inf')
patience_counter = 0
train_losses     = []
val_losses       = []

for epoch in range(1, EPOCHS + 1):

    model.train()
    running_loss = 0.0
    for imgs, cmds, angles in train_loader:
        imgs   = imgs.to(device, non_blocking=True)
        cmds   = cmds.to(device, non_blocking=True)
        angles = angles.to(device, non_blocking=True)
        optimizer.zero_grad()
        preds  = model(imgs, cmds)
        loss   = criterion(preds, angles)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, cmds, angles in val_loader:
            imgs   = imgs.to(device, non_blocking=True)
            cmds   = cmds.to(device, non_blocking=True)
            angles = angles.to(device, non_blocking=True)
            preds  = model(imgs, cmds)
            val_loss += criterion(preds, angles).item()
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    print(f'Epoch {epoch:3d}/{EPOCHS}  train={train_loss:.5f}  val={val_loss:.5f}  lr={optimizer.param_groups[0]["lr"]:.2e}')

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'cil_model2_best.pth')
        print(f'  ✓ Mejor modelo guardado (val_loss={best_val_loss:.5f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch}')
            break

print(f'\nEntrenamiento completo. Mejor val_loss: {best_val_loss:.5f}')

---
## 11. Curva de entrenamiento

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss', color='#2196F3')
plt.plot(val_losses,   label='Val Loss',   color='#F44336')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Curva de entrenamiento CIL — Notebook 31')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()

---
## 12. Exportar modelo como TorchScript — `cil_model2.pt`

Se usa `torch.jit.trace` con un ejemplo de batch_size=1 y `nav_command` como `LongTensor`.  
El controlador V2.0 llama `model(tensor, torch.tensor([cmd], dtype=torch.long))` — forma compatible.

> **Nota:** `gather` es una operación tensorial (no flujo de control Python), por lo que `trace`
> captura correctamente la selección de rama para cualquier valor de `nav_command` en runtime.

In [ ]:
model.load_state_dict(torch.load('cil_model2_best.pth', map_location=device))
model.eval()

example_img  = torch.randn(1, 3, IMG_H, IMG_W, device=device)
example_cmd  = torch.tensor([0], dtype=torch.long, device=device)
traced_model = torch.jit.trace(model, (example_img, example_cmd))
traced_model.save('cil_model2.pt')

print('Modelo exportado como cil_model2.pt')
print('Copia este archivo a la carpeta models/ del proyecto Webots (controlador V2.0)')

loaded   = torch.jit.load('cil_model2.pt', map_location='cpu')
loaded.eval()
test_img = torch.randn(1, 3, IMG_H, IMG_W)
print('\nSmoke test (todos los comandos):')
for cmd_id, cmd_name in CMD_NAMES.items():
    test_cmd = torch.tensor([cmd_id], dtype=torch.long)
    test_out = loaded(test_img, test_cmd)
    print(f'  CMD {cmd_id} ({cmd_name:11s}): {test_out.item():.4f} rad')

---
## 13. Descargar archivos desde Colab

In [ ]:
from google.colab import files
files.download('cil_model2.pt')
files.download('training_curve.png')

---
## 14. Error absoluto medio por comando

Evaluación final sobre el conjunto de validación. Objetivos orientativos:

| Comando | MAE objetivo |
|---------|-------------|
| LANE_FOLLOW | < 0.05 rad |
| STRAIGHT | < 0.04 rad |
| LEFT | < 0.10 rad |
| RIGHT | < 0.10 rad |

In [ ]:
model.eval()
cmd_errors = {0: [], 1: [], 2: [], 3: []}

with torch.no_grad():
    for imgs, cmds, angles in val_loader:
        imgs   = imgs.to(device, non_blocking=True)
        cmds   = cmds.to(device, non_blocking=True)
        angles = angles.to(device, non_blocking=True)
        preds  = model(imgs, cmds)
        errors = (preds - angles).abs().cpu().numpy()
        for i, cmd in enumerate(cmds.cpu().numpy()):
            cmd_errors[int(cmd)].append(errors[i])

print('Error absoluto medio por comando:')
for cmd, errs in cmd_errors.items():
    if errs:
        print(f'  {cmd} ({CMD_NAMES[cmd]:11s}): MAE={np.mean(errs):.4f} rad  ({len(errs)} muestras)')
    else:
        print(f'  {cmd} ({CMD_NAMES[cmd]:11s}): sin muestras en validacion')